# Teach-Back Tutor: a walkthrough

The student photographs a textbook page and **teaches it to an AI classmate who missed the class**. The classmate asks naive questions and sometimes says something wrong on purpose; the student has to catch it. Explaining an idea is the best test of understanding (the Feynman technique).

This notebook walks through the pipeline step by step. For the full experience (typing or speaking, in your own language) run the app: `streamlit run app.py`.

**Bodhan models:** `indic-ocr`, `indic-transliterate`, `indic-translate`, `indic-speak`, `indic-transcribe`. A companion LLM (any OpenAI-compatible endpoint) does the grading and the classmate's lines.

**Requires** `BODHAN_API_KEY` and `LLM_API_KEY` in `.env` (see `.env.example`).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Audio

load_dotenv(".env")  # before importing the recipe modules, which read their settings on import

import bodhan  # noqa: E402
import lesson  # noqa: E402
import memory  # noqa: E402
import parent  # noqa: E402
import tutor  # noqa: E402

## Step 1: read the page and build the lesson card

`indic-ocr` turns the photo into text. The LLM then drafts a *lesson card*: the key concepts the student should explain, and plausible misconceptions the classmate can plant. Every item must quote the page; Python drops any item whose quote is not actually on the page.

In [ ]:
page = Path("sample_data/photosynthesis_page.png")
page_text, card = lesson.card_from_image(page.read_bytes(), page.name)
print(page_text[:400], "...\n")
print("Topic:", card["topic"])
for c in card["concepts"]:
    print("  concept:", c["name"])
for m in card["misconceptions"]:
    print(f"  mistake to plant ({m['difficulty']}):", m["wrong_statement"])

## Step 2: teach it to the classmate

Every turn: the LLM **grades** the student's message (JSON), Python **updates** the scores and **chooses the next move** (ask about a gap, plant a mistake, give a hint, go deeper, wrap up), and the LLM **writes the classmate's line** for that move.

In [ ]:
s = tutor.Session(card=card, band="middle")
print(f"{s.cfg['name']}:", tutor.start(s))


def say(message):
    reply, info = tutor.step(s, message)
    print(f"[move: {info['move']} | level {info['level']} | quality {info['eval'].get('quality')}/3]")
    print(f"{s.cfg['name']}:", reply)

In [ ]:
say("We learned photosynthesis. Plants make their own food in their leaves using sunlight, water and air.")

Students can type in their own language using English letters. `indic-transliterate` converts it to the native script and `indic-translate` brings it to English for grading:

In [ ]:
hinglish = "patton mein hara rang chlorophyll hota hai jo dhoop ko pakadta hai"
native = bodhan.transliterate(hinglish, language="hi")
english = bodhan.translate(native, target_language="en")
print(native, "->", english)
say(english)

## Step 3: hear the classmate, and talk back by voice

The reply is translated to the student's language and spoken by `indic-speak` in the persona's style. In the app, the student can answer by voice; `indic-transcribe` turns it back into text (here we simply transcribe the classmate's own audio to show the round trip).

In [ ]:
reply_hi = bodhan.translate(s.history[-1]["content"], target_language="hi")
audio = bodhan.speak_long(reply_hi, "hi", style=s.cfg["voice_style"])
print(reply_hi)
first = bodhan.speak(reply_hi.split("।")[0], bodhan.VOICES["hi"], "hi")
print("heard back:", bodhan.transcribe(first, language="hi"))
Audio(audio)

## Step 4: report, memory and a note for parents

The report is computed in Python from the session state. It is saved to the student's memory, so the next session starts at the level they reached, asks about their weak spots first, and re-plants the mistakes they missed. Finally, a short note for parents is written, translated and spoken.

In [ ]:
rep = tutor.report(s)
print("Understanding:", rep["mastery"], "%  |  tricks caught:", rep["mistakes_caught"])
for c in rep["concepts"]:
    print(f"  {c['status']:12} {c['name']}")

profile = memory.load("notebook_demo")
memory.save(memory.record(profile, s, rep))
print("\nnext session will know:", memory.for_session(profile, card["topic"]))

In [ ]:
note = parent.parent_note(rep, "hi")
print(note["english"], "\n")
print(note["text"])
Audio(note["audio"])

## Wrap-up

- `streamlit run app.py`: the full app (photo, typing or voice, any of 12 languages, report, parent note).
- `python simulate.py --all`: the evaluation harness. An LLM role-plays a strong, a partial and a confused student, and we check the scores match what each one knows.
- `pytest test_policy.py`: offline tests for the decisions Python makes.